In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import csv

In [ ]:
year_groups = {
    "Group1": ['03_04', '05_06'],
    "Group2": ['07_08', '09_10'],
    "Group3": ['11_12', '13_14'],
    "Group4": ['15_16', '17_18'],
    "Group5": ['19_20', '21_23']
}


if not os.path.exists("./data_set"):
    os.makedirs("./data_set")

if not os.path.exists("./data_set/data_years"):
    os.makedirs("./data_set/data_years")

if not os.path.exists("./data_set/balance_data"):
    os.makedirs("./data_set/balance_data")

In [ ]:
all_data = pd.DataFrame()

for group_name, years in year_groups.items():

    # 1. 每次進入新群組，初始化一個空的資料表
    data = pd.DataFrame()
    for yr in years:
        tem = pd.read_csv(f'./data_set/data_years/Nhanes_cleaned_{yr}.csv')
        data = pd.concat([data, tem], axis=0)
        
    # 2. 清除包含缺失值的列
    data = data.dropna(axis=0, how='any').reset_index(drop=True)
        
    # 3. 過濾風險組與健康組
    all_risk_groups = data.query("DIQ010 == 1.0 or DIQ010 == 3.0").copy()
    all_health_groups = data.query("DIQ010 == 2.0").copy()

    # 4. 數據平衡：使用隨機抽樣 (.sample) 抽取與風險組相同數量的健康組資料
    num_risk = len(all_risk_groups)
    num_health = len(all_health_groups)
    
    if num_health > num_risk:
        all_health_groups_balanced = all_health_groups.sample(n=num_risk, random_state=42).copy()
    else:
        all_health_groups_balanced = all_health_groups.copy()

    # 5. 合併平衡後的數據
    balance_data = pd.concat([all_health_groups_balanced, all_risk_groups], axis=0)
    balance_data = balance_data.reset_index(drop=True)
    
    # 6. 充分打亂數據集順序
    balance_data = balance_data.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # 7. 儲存至指定路徑
    balance_data.to_csv(f'./data_set/balance_data/balance_data_{group_name}.csv', index=False)
    all_data = pd.concat([all_data, balance_data], axis=0)

    # 8. 印出統計結果
    risk_count = len(balance_data.query("DIQ010 == 1.0 or DIQ010 == 3.0"))
    health_count = len(balance_data.query("DIQ010 == 2.0"))
    
    print(f'{group_name}資料集:')
    print('危險的人數:', risk_count)
    print('健康的人數:', health_count)
    print('總共:', risk_count + health_count)
    print()
    
all_data.to_csv(f'./data_set/balance_data/all_balance_data.csv', index=False)

Group1資料集:
危險的人數: 486
健康的人數: 486
總共: 972

Group2資料集:
危險的人數: 358
健康的人數: 358
總共: 716

Group3資料集:
危險的人數: 696
健康的人數: 696
總共: 1392

Group4資料集:
危險的人數: 866
健康的人數: 866
總共: 1732

Group5資料集:
危險的人數: 1237
健康的人數: 1237
總共: 2474

